<h1 align = "center">
E-Commerce Data Profiling & Validation
</h1>



# 1. Objective

This profiling exercise is intended to assess the structure, quality, consistency, and analytical suitability of the Products, Sales, and Customer datasets before developing the Power BI commercial analytics model.

The profiling focuses on dataset structure, data types, missing values, duplicates, uniqueness, categorical consistency, date and time validity, numerical anomalies, referential integrity, and reconciliation of transaction-level financial fields.

**No analytical metric or Power BI measure will be finalized until the underlying field definitions and relationships have been validated.**



## 2. Environment & Data Loading

In [1]:
import pandas as pd
import numpy as np

products = pd.read_csv("../data/raw/products.csv")
sales = pd.read_csv("../data/raw/sales.csv")
customers = pd.read_csv("../data/raw/customers.csv")

For cleaner data loading, let's establish the three datsets:

In [2]:
datasets = {
    "products": products,
    "sales": sales,
    "customers": customers
}


## 3. Dataset Structure

Our first question is:

**What exactly are we working with?**

For each data frame, we want to establish:
- Number of rows
- Number of columns
- Column names
- Data types

In [3]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")


products
--------------------------------------------------
Rows: 2,000
Columns: 12

sales
--------------------------------------------------
Rows: 250,000
Columns: 21

customers
--------------------------------------------------
Rows: 40,000
Columns: 15


### Why is this important?

We are establishing the grain of each dataset. For instance, we know that:
- From products data frame, one row should represent one product
- From Customers data frame, one row represents one customer, and 
- From Sales data frame, oen row should represent one transaction/ order line

However, we are still not assuming Sales grain just yet, untill we test whether order_id is unique.

We further want to create a reusable profiling function that will free us from manually examining each dataset. 

We'll create a small function as follows:

In [4]:
def profile_structure(df, name):
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean() * 100).round(2).values,
        "unique_values": df.nunique(dropna=True).values
    })
    
    profile.insert(0, "dataset", name)
    
    return profile

With this function, we are then going to profile the datasets in one go as follows:

In [5]:
products_profile = profile_structure(products, "Products")
sales_profile = profile_structure(sales, "Sales")
customers_profile = profile_structure(customers, "Customers")

In [6]:
structure_profile = pd.concat(
    [products_profile, sales_profile, customers_profile],
    ignore_index=True
)

structure_profile

,dataset,column,dtype,non_null,missing,missing_pct,unique_values
0,Products,Product_ID,str,2000,0,0.00,2000
1,Products,Product_Name,str,2000,0,0.00,350
2,Products,Category,str,2000,0,0.00,7
3,Products,Brand,str,2000,0,0.00,33
4,Products,Original_Price,float64,2000,0,0.00,2000
5,Products,Discount_Percent,int64,2000,0,0.00,46
6,Products,Discount_Amount,float64,2000,0,0.00,1997
7,Products,Selling_Price,float64,2000,0,0.00,2000
8,Products,Stock_Quantity,int64,2000,0,0.00,855
9,Products,Weight_kg,float64,2000,0,0.00,472



## 4. Data Types



Having explicitly assessed whether fields have appropriate types, we want to explicitly convert date fields.

Dates in both customers and sales data frame have been stored as string variables instead of datetime.


In [7]:
date_columns = {
    "sales": ["Order_Date", "Delivery_Date"],
    "customers": ["Date_of_Birth", "Registration_Date"]
}

for col in date_columns["sales"]:
    sales[col] = pd.to_datetime(sales[col], errors="coerce")

for col in date_columns["customers"]:
    customers[col] = pd.to_datetime(customers[col], errors="coerce")

We use the `errors=coerce` comand to convert invalid dates to `Nat` which we can then subsequently identify


## 5. Missing Values

Here we want a proper table, and not simply a  `df.isna().sum` comand


In [8]:
def missingness_report(df, name):
    result = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })
    
    result["dataset"] = name
    
    return result.sort_values("missing_pct", ascending=False)

In [9]:
missingness = pd.concat([
    missingness_report(products, "Products"),
    missingness_report(sales, "Sales"),
    missingness_report(customers, "Customers")
])

missingness

,column,missing_count,missing_pct,dataset
Product_ID,Product_ID,0,0.00,Products
Product_Name,Product_Name,0,0.00,Products
Category,Category,0,0.00,Products
Brand,Brand,0,0.00,Products
Original_Price,Original_Price,0,0.00,Products
Discount_Percent,Discount_Percent,0,0.00,Products
Discount_Amount,Discount_Amount,0,0.00,Products
Selling_Price,Selling_Price,0,0.00,Products
Stock_Quantity,Stock_Quantity,0,0.00,Products
Weight_kg,Weight_kg,0,0.00,Products



## 6. Duplicate Records

First we must distinguish between **Exact duplicate rows**


In [10]:
for name, df in datasets.items():
    print(name, df.duplicated().sum())

products 0
sales 0
customers 0


But we also want to investigate **key duplicates**

Ideally

One `Product_ID` translates to one product; and

One `Customer_ID` translates to one customer;

### Products

In [11]:
products["Product_ID"].duplicated().sum()

np.int64(0)

### Customers

In [12]:
customers["Customer_ID"].duplicated().sum()

np.int64(0)

### Sales

This is more nuanced as a duplicate doesn't automatically insinuate an error.

A duplicate could mean that one order contains multiple products in case teh sales dataset is at teh order-line level instead of the order level. 

The distinction is crucial for calculating 
- Orders
- revenue
- quantity, and
- AOV

In [13]:
sales["Order_ID"].duplicated().sum()

np.int64(0)


## 7. Unique Values & Categorical Validation


For important categorical fields

In [14]:
categorical_columns = [
    "Category",
    "Brand"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(products[col].value_counts(dropna=False))


Category
Category
Electronics    290
Fashion        285
Home           285
Beauty         285
Sports         285
Books          285
Grocery        285
Name: count, dtype: int64

Brand
Brand
Nike          114
Puma          114
Samsung        58
HP             58
boAt           58
Noise          58
Apple          58
Levis          57
Peter          57
Titan          57
IKEA           57
Prestige       57
Philips        57
Godrej         57
Urban          57
Lakme          57
Mamaearth      57
Nivea          57
Dove           57
Plum           57
Decathlon      57
Yoga           57
Dumbbells      57
Fiction        57
NCERT          57
Comics         57
Notebook       57
Stationery     57
Aashirvaad     57
Fortune        57
Lays           57
Nescafe        57
Haldiram       57
Name: count, dtype: int64


### For Sales

In [15]:
for col in [
    "Payment_Mode",
    "Order_Status",
    "Coupon_Code"
]:
    print(f"\n{col}")
    print(sales[col].value_counts(dropna=False))


Payment_Mode
Payment_Mode
UPI            128474
COD             82093
Debit Card      28856
Credit Card     10577
Name: count, dtype: int64

Order_Status
Order_Status
Delivered     200139
Cancelled      12507
Returned       12493
Shipped        12459
Processing     12402
Name: count, dtype: int64

Coupon_Code
Coupon_Code
NaN          199815
SAVE10        24997
DIWALI100     12609
FLAT50        12579
Name: count, dtype: int64


#### For Customers

In [16]:
for col in [
    "Gender",
    "Age_Group",
    "Customer_Tier"
]:
    print(f"\n{col}")
    print(customers[col].value_counts(dropna=False))


Gender
Gender
male      20003
female    19997
Name: count, dtype: int64

Age_Group
Age_Group
26-35    14059
18-25     9876
36-45     8011
46-55     4841
56-65     2437
65+        776
Name: count, dtype: int64

Customer_Tier
Customer_Tier
Platinum    27349
Gold         6692
Silver       5959
Name: count, dtype: int64



## 8. Numerical Data Quality


We want to check teh descriptive statistics for each dataset

In [17]:
products.describe().T

,count,mean,std,min,25%,50%,75%,max
Original_Price,2000.0,18323.619210,29472.155725,52.25,1522.5050,4775.355,22358.1150,149658.64
Discount_Percent,2000.0,27.471000,13.309874,5.00,16.0000,27.000,39.0000,50.00
Discount_Amount,2000.0,5143.280140,9732.991260,6.29,332.2600,1154.915,5452.9350,71289.60
Selling_Price,2000.0,13180.339070,21514.949630,26.13,1088.0375,3543.415,16063.8125,136973.33
Stock_Quantity,2000.0,503.479000,287.140337,10.00,250.7500,504.500,749.0000,1000.00
Weight_kg,2000.0,2.640980,1.373574,0.21,1.4300,2.725,3.7900,5.00
Avg_Rating,2000.0,4.398185,0.095983,3.91,4.3400,4.400,4.4600,4.76
Total_Reviews,2000.0,60.015000,27.645161,15.00,38.0000,54.000,78.0000,139.00


In [18]:
sales.describe().T

,count,mean,min,25%,50%,75%,max,std
Order_Date,250000,2025-06-15 09:37:10.156799,2024-06-01 00:00:00,2024-12-08 00:00:00,2025-06-15 00:00:00,2025-12-22 00:00:00,2026-06-30 00:00:00,NaN
Delivery_Date,250000,2025-06-19 21:38:16.512000,2024-06-03 00:00:00,2024-12-13 00:00:00,2025-06-19 00:00:00,2025-12-27 00:00:00,2026-07-07 00:00:00,NaN
Quantity,250000.0,1.24958,1.0,1.0,1.0,1.0,3.0,0.535689
Unit_Price,250000.0,19202.477645,26.13,1551.55,7208.93,24433.27,136973.33,26720.495176
Order_Value,250000.0,23967.862192,26.13,1825.89,8411.16,28612.81,410919.99,37629.320529
Shipping_Cost,250000.0,4.911971,0.0,0.0,0.0,0.0,90.0,18.141581
Coupon_Discount,250000.0,250.001146,0.0,0.0,0.0,0.0,41092.0,1399.928532
Total_Amount,250000.0,23722.773017,-21.56,1803.9,8339.49,28344.92,410919.99,37267.644163
Rating,120030.0,4.3994,3.0,4.0,4.0,5.0,5.0,0.663915
Customer_Age,250000.0,35.309416,18.0,26.0,33.0,43.0,75.0,12.574524


In [19]:
customers.describe().T

,count,mean,min,25%,50%,75%,max,std
Age,40000.0,35.306225,18.0,26.0,33.0,43.0,75.0,12.581443
Date_of_Birth,40000,1989-09-20 05:28:08.400000,1950-01-20 00:00:00,1982-01-12 00:00:00,1992-01-10 00:00:00,1999-01-08 00:00:00,2007-01-06 00:00:00,NaN
Pincode,40000.0,549135.098375,100010.0,325689.75,547573.5,773876.0,999986.0,259430.68886
Registration_Date,40000,2023-11-29 20:07:43.680000,2023-06-01 00:00:00,2023-08-30 00:00:00,2023-11-29 00:00:00,2024-02-29 00:00:00,2024-05-31 00:00:00,NaN
Total_Orders,40000.0,5.003475,0.0,3.0,5.0,6.0,17.0,2.232485
Total_Spent,40000.0,118539.15886,0.0,37814.1475,90059.66,169958.4175,890131.55,106205.42346


Now we specifically test **business rule**

#### On Product Prices
We are checking for

In [20]:
(products["Original_Price"] < 0).sum()
(products["Selling_Price"] < 0).sum()
(products["Discount_Percent"] < 0).sum()
(products["Discount_Percent"] > 100).sum()

np.int64(0)

#### On Quantity
We are chacking for

In [21]:
(sales["Quantity"] <= 0).sum()

np.int64(0)

#### On Ratings
We are checking for

In [22]:
sales["Rating"].describe()


count    120030.000000
mean          4.399400
std           0.663915
min           3.000000
25%           4.000000
50%           4.000000
75%           5.000000
max           5.000000
Name: Rating, dtype: float64

In [23]:
products["Avg_Rating"].describe()

count    2000.000000
mean        4.398185
std         0.095983
min         3.910000
25%         4.340000
50%         4.400000
75%         4.460000
max         4.760000
Name: Avg_Rating, dtype: float64

## 9. Date & Time Validation

Here we want to check if there are invalid dates

#### Sales
We want to check when the first sale was done, and when the last sale was done in the sales data frame

We also want to check when the first and last deliveries were made according to the sales data frame

In [24]:
sales["Order_Date"].min()

Timestamp('2024-06-01 00:00:00')

In [25]:
sales["Order_Date"].max()

Timestamp('2026-06-30 00:00:00')

In [26]:
sales["Delivery_Date"].min()

Timestamp('2024-06-03 00:00:00')

In [27]:
sales["Delivery_Date"].max()

Timestamp('2026-07-07 00:00:00')

#### Customers
We want to check when the first and last customers was registered according to the Customers data frame

We also want to check the when the oldest and youngest customers were born

In [28]:
customers["Registration_Date"].min()

Timestamp('2023-06-01 00:00:00')

In [29]:
customers["Registration_Date"].max()

Timestamp('2024-05-31 00:00:00')

In [30]:
customers["Date_of_Birth"].min()

Timestamp('1950-01-20 00:00:00')

In [31]:
customers["Date_of_Birth"].max()

Timestamp('2007-01-06 00:00:00')

We also want to test whether there are any orders whose delivery dates come before order date. These would be invalid delivery dates

In [32]:
(sales["Delivery_Date"] < sales["Order_Date"]).sum()

np.int64(0)

We want then to investigate the delivery days (days between date of ordering a product and when it's delivered) to see if there are any negative values or unusually large values.

We'll first create a temporary profiling variable; we'll name it Delivery_Days

In [33]:
sales["Delivery_Days"] = (
    sales["Delivery_Date"] - sales["Order_Date"]
).dt.days

In [34]:
sales["Delivery_Days"].describe()

count    250000.000000
mean          4.500768
std           1.710942
min           2.000000
25%           3.000000
50%           5.000000
75%           6.000000
max           7.000000
Name: Delivery_Days, dtype: float64

From the output, we have the average number of days for delivery is 4.5, with the minimum being 2 days and maximum being 7 days. 

These are valid. So there is no need to test further whether there are any negative or unusually large number of days for delivery.

### Order Time Analysis
Order time shall be essential for examining purchasing patterns. 

Earlier outputs showed that Order_Time was stored as string, so we'll need the variable converted to time object.

In [35]:
sales["Order_Time"].head(20)

0     08:20:00
1     20:05:00
2     14:59:00
3     14:33:00
4     07:19:00
5     11:56:00
6     10:32:00
7     17:46:00
8     12:19:00
9     07:45:00
10    03:58:00
11    13:15:00
12    04:46:00
13    18:53:00
14    18:23:00
15    21:09:00
16    05:38:00
17    01:00:00
18    12:03:00
19    02:31:00
Name: Order_Time, dtype: str

In [36]:
sales['Order_Time'] = pd.to_datetime(sales['Order_Time'], format='%H:%M:%S',errors="coerce").dt.time


In [37]:
sales["Order_Time"].head(20)

0     08:20:00
1     20:05:00
2     14:59:00
3     14:33:00
4     07:19:00
5     11:56:00
6     10:32:00
7     17:46:00
8     12:19:00
9     07:45:00
10    03:58:00
11    13:15:00
12    04:46:00
13    18:53:00
14    18:23:00
15    21:09:00
16    05:38:00
17    01:00:00
18    12:03:00
19    02:31:00
Name: Order_Time, dtype: object

In [38]:

sales["Order_Hour"] = pd.to_datetime(sales["Order_Time"],format="%H:%M:%S",errors="coerce").dt.hour


In [39]:

sales["Order_Hour"].value_counts().sort_index()

Order_Hour
0     10451
1     10412
2     10411
3     10446
4     10467
5     10357
6     10448
7     10299
8     10659
9     10299
10    10512
11    10468
12    10436
13    10432
14    10334
15    10462
16    10335
17    10315
18    10423
19    10382
20    10475
21    10342
22    10417
23    10418
Name: count, dtype: int64


## 10. Referential Integrity


This is critical because we have relationships between our datasets

#### products - sales relationship
Here we want to check whether there are sales product IDs that don't exist in products

In [40]:
sales_products = set(sales["Product_ID"].dropna())
product_ids = set(products["Product_ID"].dropna())

missing_products = sales_products - product_ids

len(missing_products)

0

The output confirms that there are no product IDs in sales that don't exist in products data frame

### Customer - sales relationship
We want to check whether there are customer IDs that exist in sales but not in customer data frame

In [41]:
sales_customers = set(sales["Customer_ID"].dropna())
customer_ids = set(customers["Customer_ID"].dropna())

missing_customers = sales_customers - customer_ids

len(missing_customers)

0

Again, there are no customer IDs existing in sales data frame that don't exist in customer data frame


## 11. Transaction-Level Financial Reconciliation


This is particularly essential. We want to understand what the following fields mean:
- Quantity
- Unit_Price
- Order_Value
- Shipping_Cost
- Coupon_Discount, and
- Total_Amount

We'll start with order value

#### Order_Value

We'll create a calculated Order_Value field, which will be teh product of the unit price and quantity sold

In [42]:
sales["calculated_order_value"] = (
    sales["Quantity"] * sales["Unit_Price"]
)

We'll then find teh difference between the calculated_order_value field and our Order_Value field initially calculated.

We are interested in seeing whether the Order_Value variable is essentially determined by multiplying the product quantity with unit price.

In [43]:
sales["order_value_difference"] = (
    sales["Order_Value"] -
    sales["calculated_order_value"]
)

In [44]:
sales["order_value_difference"].describe()

count    2.500000e+05
mean     2.605776e-14
std      1.830561e-12
min     -2.910383e-11
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.820766e-11
Name: order_value_difference, dtype: float64

In [45]:
(sales["order_value_difference"].abs() < 0.01).mean()

np.float64(1.0)

Our output above shows that all pre-calculated Order_Value entries were exclusively obtainedby multiplying the Quantity with Unit_Price.

This tells us that 100 percentage of transactions satisfy:

$$ OrderValue≈Quantity×UnitPrice $$

#### Total Amount

We are interested in knowing what percentage of Total Amount records satisfy the calculation:

$$ TotalAmount=OrderValue+ShippingCost−CouponDiscount $$

We'll create candidate formula

In [46]:
sales["calc_total_a"] = (
    sales["Order_Value"]
    + sales["Shipping_Cost"]
    - sales["Coupon_Discount"]
)

In [47]:
sales["difference_a"] = (
    sales["Total_Amount"] -
    sales["calc_total_a"]
)

In [48]:
(sales["difference_a"].abs() < 0.01).mean()

np.float64(1.0)

With 100 percent of records satifying our calculation we are now confident that 

$$ TotalAmount=OrderValue+ShippingCost−CouponDiscount $$

If formula doesn't reconcille, it's important to test alternative interpretations


## 12. Cross-Dataset Consistency


Here we want to cross-check customer aggregates 

Our dataset contains Total_Orders and Total_Spent that are potentially derived from sales

We, therefore, want to come up with transaction-level cuatomer summaries

In [49]:
sales_customer_summary = (
    sales.groupby("Customer_ID")
    .agg(
        calculated_orders=("Order_ID", "nunique"),
        calculated_spent=("Total_Amount", "sum")
    )
    .reset_index()
)

We then merge 

In [50]:
customer_validation = customers.merge(
    sales_customer_summary,
    on="Customer_ID",
    how="left"
)

Now we compare

In [51]:
customer_validation["orders_difference"] = (
    customer_validation["Total_Orders"] -
    customer_validation["calculated_orders"]
)

And

In [52]:
customer_validation["spent_difference"] = (
    customer_validation["Total_Spent"] -
    customer_validation["calculated_spent"]
)

In [53]:
sales_customer_summary.head()

,Customer_ID,calculated_orders,calculated_spent
0,CUST00000001,5,40098.48
1,CUST00000002,10,96487.46
2,CUST00000003,2,73387.76
3,CUST00000004,9,388923.23
4,CUST00000005,6,75252.85


## 13. Data Profiling Summary

Check	                    |Result	|Status	|Action
----------------------------|-------|-------|------------
Product IDs unique	        |100%	|Pass	|Use as product key
Customer IDs unique	        |100%	|Pass	|Use as customer key
Missing product IDs	        |0%	    |Pass	|—
Missing customer IDs	    |0%	    |Pass	|—
Invalid delivery dates	    |0	    |Pass	|—
Order Value reconciliation	|100%	|Pass	|—
Total Amount reconciliation	|100%	|Pass	|Adopt formula
Customer totals reconcile	|100%	|Pass	|—